# Module 2.1: Operating System Evaluation & Replacing Unlicensed Windows 11 with Ubuntu Server 24.04 LTS

## Homelab Expansion (Chapter 2) - Step 2: Operating System & Headless Laptop Tuning

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why wiping the laptop's existing **unlicensed Windows 11** installation in favor of a headless Linux server OS is mandatory for 24/7 GPU homelab stability.
2. Evaluate Linux server operating systems (`Ubuntu Server 24.04 LTS`, `Debian 12 Bookworm`, and `Proxmox VE 8`) for an ASUS ROG laptop with hybrid AMD/Intel + NVIDIA graphics.
3. Flash a bootable `Ubuntu Server 24.04 LTS (64-bit x86_64)` USB installer and boot the ROG Zephyrus into the live setup environment.
4. Wipe the Windows NTFS partitions, allocate the full internal NVMe SSD to `ext4`, and provision the host as `homelab-gpu`.
5. Record your permanent Node 2 credentials for use across the rest of the homelab curriculum.

## 2.1.1 Conceptual Background: Why Wipe Unlicensed Windows 11 & Which OS Should You Run?

Right now, your 2019 ROG Zephyrus is running an **unlicensed copy of Windows 11**. While it might be tempting to install Docker Desktop on Windows and call it a day, running an unlicensed desktop Windows OS as a 24/7 headless server introduces crippling operational flaws:

### 1. Why Unlicensed Windows 11 Fails as a 24/7 Homelab Server
- **3.5 GB+ Idle RAM & VRAM Overhead:** Windows 11 Desktop (`dwm.exe`, Windows Defender, telemetry, and WSL2's `vmmem` hypervisor VM) consumes **3.5 GB to 5.0 GB of RAM** and **400 MB+ of GPU VRAM** before a single container is even launched.
- **Forced Unattended Reboots & Activation Nags:** Windows 11 automatically restarts outside "Active Hours" for cumulative updates and driver pushes, killing your Docker stacks or getting stuck at a login screen waiting for user interaction. Unlicensed installs also lock out personalization/group policy controls and display persistent activation watermarks.
- **WSL2 Network & GPU Paravirtualization Quirks:** Docker Desktop on Windows runs inside a lightweight Hyper-V utility VM. Exposing container ports to your LAN or Tailscale mesh requires brittle `netsh interface portproxy` rules, and GPU sleep states are poorly managed.

### 2. Operating System Candidate Comparison for a 2019 ROG Zephyrus (`x86_64` + NVIDIA GPU)

| Operating System | Recommended Level | Idle RAM | NVIDIA CUDA / Docker Support | ASUS ROG Laptop Compatibility (`asus-nb-wmi`) | Verdict |
| :--- | :---: | :---: | :--- | :--- | :--- |
| **Ubuntu Server 24.04 LTS (Noble Numbat)** | 🥇 **Primary Choice** | **~320 MB** | **Best-in-Class:** Built-in `ubuntu-drivers` auto-detects NVIDIA GTX 1660 Ti / RTX 20-series GPUs and installs tested headless drivers (`nvidia-driver-550-server`) in one command. Official NVIDIA Container Toolkit apt repo support. | Modern Linux `6.8+` HWE kernel includes native `asus-nb-wmi` thermal profiles, fan curve telemetry, and Realtek 2.5GbE USB/PCIe drivers out of the box. | **Ideal choice:** Zero desktop bloat, rock-solid 5-year LTS support, and effortless NVIDIA GPU passthrough to Docker Compose. |
| **Debian 12 (Bookworm) Netinst** | 🥈 **Runner-Up** | **~180 MB** | Requires manually enabling `non-free-firmware` and `contrib` repos; older default `6.1` kernel lacks newer ASUS ROG WMI power-profile patches unless backports are configured. | Good, but Wi-Fi / USB Ethernet chipsets often require non-free firmware ISOs during install. | Great if you want strict parity with Raspberry Pi OS, but takes extra manual driver work for NVIDIA + ASUS hardware. |
| **Proxmox VE 8.x (Type-1 Hypervisor)** | 🥉 **Alternative (If You Need VMs)** | **~1.2 GB** | Passing a laptop's **hybrid Optimus GPU** (where the display is wired to AMD/Intel iGPU and the NVIDIA dGPU is a render-only 3D controller) into a VM via PCIe VFIO IOMMU passthrough is notoriously complex and locks the GPU to one VM. | Strips out standard laptop power management utilities unless configured manually on the Debian host shell. | Overkill if your primary goal is running Docker containers (Immich ML, Ollama, Jellyfin) with shared GPU access. |

> 💡 **Engineering Conclusion:** By installing **Ubuntu Server 24.04 LTS (Headless, No GUI)** directly on bare metal, your 16GB/32GB ROG Zephyrus boots in 8 seconds, uses less than **350 MB of RAM** at idle, keeps the NVIDIA GPU in its ultra-low-power **P8 idle state (~3W–5W)** when no jobs are running, and shares the GPU seamlessly across multiple Docker containers simultaneously!

## 2.1.2 Step-by-Step Laboratory Guide

### Task 2.1.1: Download Ubuntu Server 24.04 LTS & Flash USB Installer
1. On your primary workstation, download the official **Ubuntu Server 24.04 LTS (64-bit PC `amd64` ISO)** from [ubuntu.com/download/server](https://ubuntu.com/download/server).
2. Download and open **[BalenaEtcher](https://etcher.balena.io/)** (macOS/Windows/Linux) or **[Rufus](https://rufus.ie/)** (Windows).
3. Insert an empty **8GB+ USB 3.0 flash drive** into your workstation.
4. Select the `ubuntu-24.04.x-live-server-amd64.iso` image, select your USB drive, and click **Flash!**.

### Task 2.1.2: Boot the Installer & Wipe Windows 11
1. Plug your **Gigabit Ethernet cable** into the ROG Zephyrus (either into the native RJ45 port on `GX502` or via your USB 3.0 Gigabit Ethernet adapter on `GA502`/`GU502`).
2. Insert the flashed Ubuntu Server USB drive into a USB port on the ROG Zephyrus.
3. Power on the laptop and rapidly tap **`ESC`** (the ASUS ROG One-Time Boot Menu key) until the **Please select boot device** popup appears.
4. Select **`UEFI: <Your USB Drive Name>, Partition 1`** and press `Enter`, then select **Try or Install Ubuntu Server**.
5. Walk through the text-based `subiquity` installer using your arrow keys, `Space` (to toggle checkboxes), and `Enter`:
   - **Language & Keyboard:** `English (US)`.
   - **Installation Type:** Select **`Ubuntu Server`** (standard, not minimized, so common diagnostic utilities like `htop`, `curl`, and `tmux` are pre-installed).
   - **Network Connections:** Confirm your wired Ethernet interface (`enp...` or `enx...`) has automatically acquired an IPv4 lease via DHCP.
   - **Guided Storage Configuration (Wiping Windows 11):**
     - Select **`Use an entire disk`** and select your internal NVMe SSD (`/dev/nvme0n1`).
     - **Uncheck** `Set up this disk as an LVM group` (or if you leave LVM checked, on the next screen edit the `ubuntu-lv` logical volume and change its size from the default `100G` to the **maximum available capacity** so your NVMe SSD isn't left half-unallocated!). Unchecking LVM creates a clean, single full-disk `ext4` root partition (`/`) just like our Raspberry Pi setup.
     - Confirm the destructive action (`Continue`) to permanently erase the unlicensed Windows 11 partitions.
   - **Profile Setup:**
     - **Your name:** `pi` *(or your preferred admin name)*
     - **Your server's name (Hostname):** `homelab-gpu`
     - **Pick a username:** `pi` *(Keeping the admin username identical to `homelab-pi` simplifies SSH config and Ansible/scripts across your cluster!)*
     - **Password:** Choose a strong administrator password.
   - **Ubuntu Pro Upgrade:** Select `Skip for now`.
   - **SSH Setup:** Press `Space` to check **`[X] Install OpenSSH server`**.
   - **Featured Server Snaps:** **Do NOT check anything here!** *(CRITICAL: Never install `docker` from the Snap list during Ubuntu setup—Snap Docker has strict confinement bugs with NVIDIA GPUs and custom bind mounts. We will install official upstream `docker-ce` via `apt` in Module 4.1).*
6. Wait ~3 minutes for installation to finish, select **`Reboot Now`**, remove the USB flash drive when prompted, and press `Enter`.

---

## 📝 Node 2 Credential Record Sheet

*Record the exact credentials configured for your ROG Zephyrus GPU node below:*

> ### 🔑 Node 2 (`homelab-gpu`) Credentials
> - **Hostname:** `homelab-gpu`
> - **Admin Username:** `pi`
> - **Password:** `[Write your password here]`
> - **Local mDNS Address:** `homelab-gpu.local`
> - **Target Reserved Static IP (Step 3):** `10.0.0.201` *(paired alongside `homelab-pi` at `10.0.0.200`)*

### Task 2.1.3: Verify Clean Bare-Metal Boot, Storage Allocation & RAM Footprint
Once the laptop reboots into the Ubuntu Server terminal prompt (`homelab-gpu login:`), either log in locally or connect via SSH from your workstation:
```bash
ssh pi@homelab-gpu.local
```
Run the following verification commands to confirm the unlicensed Windows 11 installation is gone, the full NVMe SSD is mounted, and baseline memory usage is under 400 MB:
```bash
# 1. Verify OS version, kernel, and x86_64 architecture
hostnamectl

# 2. Confirm the full internal NVMe SSD is allocated to / (and no Windows NTFS partitions remain)
lsblk -f
df -h /

# 3. Verify ultra-low headless RAM consumption (~300MB used out of 16GB/32GB!)
free -h

# 4. Confirm the kernel sees both the CPU and the discrete NVIDIA PCIe GPU
lspci | grep -E "VGA|3D"
```

### 🖼️ Task 2.1.3 Verification: Ubuntu Server 24.04 LTS Bare-Metal Verification

**Verification Objective:** Capture a screenshot of your terminal session on `homelab-gpu` showing the output of `hostnamectl`, `df -h /`, `free -h`, and `lspci | grep -E "VGA|3D"`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Unlicensed Windows 11 completely wiped from the internal NVMe SSD.
- [ ] Ubuntu Server 24.04 LTS (`x86_64`) installed with full NVMe disk capacity allocated to `/`.
- [ ] OpenSSH server enabled (and Snap Docker intentionally skipped).
- [ ] Discrete NVIDIA GPU verified on the PCIe bus via `lspci | grep -E "VGA|3D"`.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 2.2: Clamshell Lid-Close Mode, Screen Blanking & Power Capping](./02_2_headless_lid_close_and_power_tuning.ipynb).